<a href="https://colab.research.google.com/github/JosephMacula/math_and_AI_seminar/blob/main/llm_project/notebooks/llm_proj_step3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%load_ext autoreload
%autoreload 2

In [2]:
class Value:
    def __init__(self, data, _parents=()):
        self.data = float(data)
        self.grad = 0.0
        self._parents = _parents
        self._backward = lambda: None

    def __repr__(self):
        return f'Value({self.data:.4f}, grad={self.grad:.4f})'

    def _wrap(self, other):
        return other if isinstance(other, Value) else Value(other)

    def __add__(self, other):
        other = self._wrap(other)
        out = Value(self.data + other.data, (self, other))
        def _backward():
            self.grad  += out.grad          # d(a+b)/da = 1
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = self._wrap(other)
        out = Value(self.data * other.data, (self, other))
        def _backward():
            self.grad  += other.data * out.grad   # d(ab)/da = b
            other.grad += self.data  * out.grad
        out._backward = _backward
        return out

    def __pow__(self, k):
        assert isinstance(k, (int, float))
        out = Value(self.data ** k, (self,))
        def _backward():
            self.grad += k * self.data ** (k - 1) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        import math
        t = math.tanh(self.data)
        out = Value(t, (self,))
        def _backward():
            self.grad += (1 - t * t) * out.grad   # d tanh = 1 - tanh^2
        out._backward = _backward
        return out

    def exp(self):
        import math
        e = math.exp(self.data)
        out = Value(e, (self,))
        def _backward():
            self.grad += e * out.grad
        out._backward = _backward
        return out

    def log(self):
        import math
        out = Value(math.log(self.data), (self,))
        def _backward():
            self.grad += (1.0 / self.data) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(self.data, 0.0), (self,))
        def _backward():
            self.grad += (1.0 if self.data > 0 else 0.0) * out.grad   # subgradient 0 at the corner
        out._backward = _backward
        return out

    # conveniences
    def __neg__(self):       return self * -1
    def __sub__(self, o):    return self + (-self._wrap(o))
    def __radd__(self, o):   return self + o
    def __rsub__(self, o):   return (-self) + o
    def __rmul__(self, o):   return self * o
    def __truediv__(self, o): return self * (self._wrap(o) ** -1)

    def backward(self):
        order, visited = [], set()
        def visit(v):
            if id(v) not in visited:
                visited.add(id(v))
                for p in v._parents:
                    visit(p)
                order.append(v)      # parents appended before children
        visit(self)
        self.grad = 1.0              # dL/dL = 1
        for v in reversed(order):    # reverse topological order
            v._backward()

In [3]:
import math


In [4]:
a = Value(2)
b = Value(-1)
L = (a*b + a)*b.tanh()
L.backward()
print(a.grad)
print(b.grad)

0.0
-1.5231883119115297


In [5]:
x1 = Value(.5)
x2 = Value(.25)
gamma = Value(4)
L_dmnd_net = (((x1 + x2 - 1)*gamma*2).exp() + 1).log()
L_dmnd_net.backward()
print(x1.grad)
print(x2.grad)

0.9536233761769404
0.9536233761769404


In [6]:
import torch

In [7]:
from proj_utils import diamond_net

In [8]:
point = torch.tensor([[0.5, 0.25]], dtype=torch.float64,
                     requires_grad=True)
_, _, _, probs = diamond_net(point, gamma=4.0)
loss = -probs[0, 1].log()             # target = inside
loss.backward()

expected = 2 * 4.0 * probs.detach()[0, 0]
print(point.grad)                      # both entries ≈ 0.9536
print(expected)

tensor([[0.9536, 0.9536]], dtype=torch.float64)
tensor(0.9536, dtype=torch.float64)
